# Tutorial Completo de `clipspy`: Sistemas Expertos en Python

Este cuaderno está diseñado como guía de estudio y material docente para aprender e impartir **CLIPS** (*C Language Integrated Production System*) integrado en **Python** mediante la librería `clipspy`.

### Contenido del cuaderno:
1. **Instalación y Configuración del Entorno** (Logs y funciones de inspección)
2. **Fundamentos: Hechos Ordenados y Plantillas (`deftemplate`)**
3. **Reglas (`defrule`), Patrones y Variables**
4. **Ejecución Paso a Paso y Depuración de la Agenda**
5. **Control de Flujo: Prioridades (`salience`) y Freno de Emergencia (`halt`)**
6. **Caso Práctico Completo: Gestión de Préstamos en Biblioteca Universitaria**
7. **Buenas Prácticas y Limpieza de Memoria**

## 1. Instalación y Configuración del Entorno

Si aún no tienes instalada la librería `clipspy`, ejecuta la siguiente celda (descomentando la línea de `pip`).

In [ ]:
# !pip install clipspy

### Configuración del Entorno y Funciones Auxiliares de Inspección
Por defecto, CLIPS envía sus mensajes internos (como `(printout t ...)`) a la salida estándar de C, que no siempre se captura directamente en las celdas de Jupyter. Para solucionarlo, conectamos un `LoggingRouter` de `clipspy` al sistema de `logging` de Python.

Además, definimos tres funciones útiles (`print_facts`, `print_rules`, `print_agenda`) y las registramos dentro del motor CLIPS usando `env.define_function()` para poder invocarlas tanto desde Python como desde dentro de las reglas de CLIPS.

In [ ]:
import clips
import logging

def crear_entorno():
    """
    Inicializa un nuevo entorno de trabajo independiente en CLIPS, configura el sistema
    de logs de Python y redirige las salidas internas de CLIPS hacia la consola.
    """
    env = clips.Environment()
    logging.basicConfig(level=10, format='%(message)s', force=True)
    router = clips.LoggingRouter()
    env.add_router(router)
    return env

def print_facts():
    """Muestra todos los hechos presentes en la Memoria de Trabajo."""
    print("--- MEMORIA DE TRABAJO (HECHOS) ---")
    for f in env.facts():
        print(f"f-{f.index}: {f}")

def print_rules():
    """Muestra todas las reglas compiladas en la Base de Conocimiento."""
    print("--- BASE DE CONOCIMIENTO (REGLAS) ---")
    for r in env.rules():
        print(r)

def print_agenda():
    """Muestra las activaciones pendientes en la Agenda."""
    print("--- AGENDA DE ACTIVACIONES ---")
    activaciones = list(env.activations())
    if not activaciones:
        print("(Agenda vacía)")
    for a in activaciones:
        print(a)

# Creamos el entorno global y registramos las funciones puente
env = crear_entorno()
env.define_function(print_facts)
env.define_function(print_rules)
env.define_function(print_agenda)

## 2. Fundamentos: Hechos Ordenados y Plantillas (`deftemplate`)

En programación declarativa no damos instrucciones paso a paso; modelamos el estado del mundo mediante **hechos** (*facts*).

1. **Hechos ordenados:** Listas simples donde importa la posición, ej. `(semaforo rojo)`.
2. **Hechos estructurados (`deftemplate`):** Equivalentes a registros o clases con campos con nombre (*slots*) y restricciones de tipo.

**Concepto clave:** `env.build(...)` compila la definición en el motor, pero los hechos definidos dentro de `deffacts` solo entran realmente en la Memoria de Trabajo cuando ejecutamos `env.reset()`.

In [ ]:
env.clear() # Limpiamos cualquier definición previa

# Definimos una plantilla estructurada para un sensor
plantilla_sensor = """
(deftemplate sensor
   (slot ubicacion (type SYMBOL))
   (slot temperatura (type FLOAT INTEGER))
   (slot estado (type SYMBOL) (allowed-values normal alerta critico) (default normal))
)
"""

# Definimos hechos iniciales (tanto ordenados como basados en plantilla)
hechos_iniciales = """
(deffacts estado-planta
   (turno manana)
   (sensor (ubicacion Sala-Servidores) (temperatura 42.5))
   (sensor (ubicacion Laboratorio) (temperatura 22.0))
)
"""

env.build(plantilla_sensor)
env.build(hechos_iniciales)

# Materializamos los hechos en la Memoria de Trabajo
env.reset()
print_facts()

## 3. Reglas (`defrule`), Patrones y Variables

Una regla tiene dos partes separadas por `=>`:
- **LHS (Left-Hand Side):** Condiciones que deben cumplirse en la Memoria de Trabajo.
- **RHS (Right-Hand Side):** Acciones a ejecutar (`assert`, `modify`, `retract`, `printout`, etc.).

Observa cómo capturamos una referencia al hecho completo mediante `?s <- (sensor ...)` para poder actualizar su estado con `(modify ?s ...)` sin provocar un bucle infinito.

In [ ]:
regla_sobrecalentamiento = """
(defrule detectar-sobrecalentamiento
   ?s <- (sensor (ubicacion ?lugar) (temperatura ?temp) (estado normal))
   (test (> ?temp 35.0))
   =>
   (modify ?s (estado critico))
   (assert (alerta-seguridad ?lugar))
   (printout t "[AVISO] Temperatura crítica (" ?temp "°C) detectada en " ?lugar crlf)
)
"""

regla_notificar_vigilante = """
(defrule notificar-vigilante
   (alerta-seguridad ?edificio)
   =>
   (printout t "[ACCIÓN] Vigilante notificado del edificio: " ?edificio crlf)
   (halt)
)
"""

env.build(regla_sobrecalentamiento)
env.build(regla_notificar_vigilante)

# Reiniciamos para que la red Rete evalúe los hechos iniciales contra las nuevas reglas
env.reset()
print_agenda()

## 4. Ejecución Paso a Paso (En Cámara Lenta)

Usar `env.run(1)` permite disparar **una única regla** de la Agenda cada vez. Esto es ideal en clase para ver cómo la inferencia de una regla modifica la memoria de trabajo y activa nuevas reglas en la Agenda.

In [ ]:
print(">>> Ejecutando PASO 1:")
env.run(1)
print()
print_agenda()
print()
print_facts()

In [ ]:
print(">>> Ejecutando PASO 2:")
env.run(1)
print()
print_agenda()

## 5. Caso Práctico Completo: Gestión de una Biblioteca Universitaria

Implementamos el sistema basado en reglas para gestionar préstamos, detectar retrasos, bloquear usuarios reincidentes y autorizar nuevos préstamos.

### Reglas del dominio:
- **R1:** Si un usuario tiene un `prestamoVencido`, registrar una `incidencia`.
- **R2:** Si un usuario acumula dos incidencias de libros distintos, marcarlo como `bloqueado`.
- **R3:** Si está `bloqueado`, notificar por pantalla.
- **R4:** Si solicita un libro `disponible` y **no** está `bloqueado`, autorizar el préstamo.
- **R5:** Si se autoriza el préstamo, cambiar el estado del libro a `prestado`.

**Detalle docente clave (`salience`):** Añadimos `(declare (salience -10))` en **R4** (`autorizar-prestamo`) para garantizar que el sistema evalúe primero todas las incidencias y bloqueos antes de conceder nuevos préstamos.

In [ ]:
env.clear()

# 1. Base de Hechos Iniciales
bh_biblioteca = """
(deffacts biblioteca
   ; Préstamos vencidos
   (prestamoVencido Ana L1)
   (prestamoVencido Ana L2)
   (prestamoVencido Luis L3)
   ; Estado de los libros
   (libro L1 prestado)
   (libro L2 prestado)
   (libro L3 prestado)
   (libro L4 disponible)
   (libro L5 disponible)
   ; Solicitudes actuales
   (solicitud Marta L4)
   (solicitud Ana L5)
)
"""

# 2. Reglas de negocio
r1 = """
(defrule R1-registrar-incidencia
   (prestamoVencido ?usuario ?libro)
   =>
   (assert (incidencia ?usuario ?libro))
)
"""

r2 = """
(defrule R2-bloquear-usuario
   (incidencia ?usuario ?libro1)
   (incidencia ?usuario ?libro2)
   (test (neq ?libro1 ?libro2))
   =>
   (assert (bloqueado ?usuario))
)
"""

r3 = """
(defrule R3-notificar-bloqueo
   (bloqueado ?usuario)
   =>
   (printout t "NOTIFICACIÓN -> Usuario " ?usuario ": cuenta bloqueada por acumulación de incidencias." crlf)
)
"""

r4 = """
(defrule R4-autorizar-prestamo
   (declare (salience -10))
   (solicitud ?usuario ?libro)
   (libro ?libro disponible)
   (not (bloqueado ?usuario))
   =>
   (assert (prestamoAutorizado ?usuario ?libro))
   (printout t "PRÉSTAMO AUTORIZADO -> " ?usuario " puede llevarse el libro " ?libro crlf)
)
"""

r5 = """
(defrule R5-actualizar-disponibilidad
   (prestamoAutorizado ?usuario ?libro)
   ?l <- (libro ?libro disponible)
   =>
   (retract ?l)
   (assert (libro ?libro prestado))
)
"""

# Compilamos todo en el entorno
env.build(bh_biblioteca)
for regla in [r1, r2, r3, r4, r5]:
    env.build(regla)

# Reiniciamos y ejecutamos el motor completo
env.reset()
print("=== EJECUTANDO MOTOR DE INFERENCIA ===")
env.run()
print("======================================\n")

print_facts()

## 6. Limpieza Ordenada de Memoria

Al finalizar nuestras pruebas en Jupyter, es buena práctica vaciar explícitamente el entorno de C de CLIPS con `env.clear()` para evitar avisos de liberación de memoria (`[ENVRNMNT8]`) cuando Python cierre el proceso.

In [ ]:
env.clear()
print("Entorno de CLIPS limpiado correctamente.")